# Explanation

This notebook represents the culmination of 28 iterative model versions developed over the course of the WiDS Datathon 2026. Early experiments (V1-V14) tested single models, feature sets, and ensemble approaches, producing scores in the 0.95-0.97 range but without a clear path to improvement. The breakthrough came when we recognized a structural property of the data: every fire within 5,000 meters of a weather station was contained, and every far fire was not. This split the problem into two distinct questions: ranking fires by risk, and timing when close fires would be contained. Answering both required two separate models rather than one. V15 introduced that two-stage architecture, V21 replaced the timing model with a survival-appropriate Coxnet and produced the single largest leaderboard jump in the entire experiment (+0.00240), and V28 added 5-seed averaging to stabilize the calibration step on the 69-fire close-fire subset. Everything after V28 (looser regularization, new features, alternative timing models) improved training scores but degraded leaderboard performance, confirming that V28 had found the right balance between model complexity and the available data.

V28 has a two-stage approach: instead of treating every fire the same, it prioritizes "close fires", those within 5,000 meters that are most likely to cause real-world impact. While it uses a broad "backbone" model to understand fire behavior generally, it layers on a specialized timing model specifically for those high-stakes, close-range threats that define the success of the project.

Under the hood, the system relies on a heavy-duty blend of three survival models, with a specific architecture called Coxnet doing about 78% of the heavy lifting. To ensure the results aren't just a fluke, the model for close-range fires is averaged across five different random "seeds" to smooth out any statistical noise. These two stages are fused together using a carefully tuned mixing parameter (β=0.55), and the final probabilities are polished using calibration techniques like Platt scaling to ensure the numbers actually reflect reality rather than just raw data.

V28 is the top-performing submission of all submissions we made, hitting a public leaderboard score of 0.97458 during the competition. Note that this was the non-final score, evaluated on a held-out subset of the test set during the competition period. The final private leaderboard score — evaluated on the full test set after the competition closed — was 0.97272, which placed us 2nd overall. Beyond the high score, the model is logically sound by ensuring "monotonicity," meaning the predicted risk of a fire hitting a community only stays the same or increases as time goes on, never confusingly dropping. It's a fully reproducible, battle-tested setup that prioritizes the fires that matter most without losing sight of the bigger picture.

# Features

## Base Features
These 22 columns are taken directly from the dataset as provided.

- **num_perimeters_0_5h**: The number of distinct fire perimeter snapshots captured in the first half-hour, reflecting how well the fire's early behavior was observed.
- **dt_first_last_0_5h**: The time gap between the first and last perimeter observation in the opening half-hour, indicating the length of the observation window.
- **low_temporal_resolution_0_5h**: A flag marking fires where observations came in at a low frequency, meaning early behavior is less precisely known.
- **area_first_ha**: The initial size of the fire in hectares at the time of first observation.
- **area_growth_rate_ha_per_h**: How quickly the fire is growing in area, measured in hectares per hour.
- **log1p_area_first**: A log-scaled version of the initial fire area, reducing the influence of extreme sizes.
- **log1p_growth**: A log-scaled version of the area growth rate, stabilizing the signal from very fast-growing fires.
- **radial_growth_rate_m_per_h**: How quickly the fire's outer edge is expanding outward in meters per hour.
- **area_growth_rel_0_5h**: The relative change in fire area over the first half-hour, capturing early growth momentum.
- **log_area_ratio_0_5h**: The log ratio of fire area between the first and last observation in the opening half-hour.
- **centroid_speed_m_per_h**: How fast the center of the fire is moving across the ground in meters per hour.
- **spread_bearing_sin**: The sine component of the direction the fire is primarily spreading toward, encoded for circular continuity.
- **spread_bearing_cos**: The cosine component of the fire's primary spread direction, paired with the sine for full directional representation.
- **centroid_displacement_m**: The total distance the fire's center has shifted from its starting point in meters.
- **closing_speed_m_per_h**: The rate at which the fire is approaching the nearest weather station in meters per hour (negative means moving away).
- **closing_speed_abs_m_per_h**: The absolute closing speed, capturing overall lateral movement regardless of direction relative to the station.
- **projected_advance_m**: The expected distance the fire will advance toward the station based on its current trajectory.
- **dist_fit_r2_0_5h**: How well a linear model fits the fire's distance trend over the first half-hour, with higher values indicating more predictable movement.
- **dist_accel_m_per_h2**: The acceleration of the fire's approach, indicating whether it is speeding up or slowing down toward the station.
- **dist_slope_ci_0_5h**: The slope of the distance trend over the first half-hour, summarizing whether the fire is generally advancing or retreating.
- **alignment_cos**: How directly the fire is spreading toward the station, measured as the cosine of the angle between spread direction and station direction.
- **alignment_abs**: The absolute alignment between fire spread direction and the direction of the nearest station, regardless of orientation.

---

## Engineered Features
These 20 columns are derived from the base features in the `engineer()` function.

- **log_dist_min**: Log-transformed minimum distance from the fire to the nearest weather station, compressing the wide range of distances into a more model-friendly scale.
- **log_dist_sq**: The square of the log distance, amplifying the distinction between fires that are very close and those that are moderately far.
- **dist_close_5k**: A binary indicator equal to 1 if the fire is within 5,000 meters of a station, the threshold that defines the close-fire subset central to this model.
- **dist_close_30k**: A binary indicator equal to 1 if the fire is within 30,000 meters of a station, capturing a broader near-threat zone.
- **tti_h**: Estimated time-to-impact in hours, computed by dividing current distance by closing speed (capped at 500 hours for static or retreating fires).
- **hazard_proxy**: A composite threat score combining fire alignment, closing speed, and distance to approximate how imminently dangerous the fire is.
- **is_dynamic**: A binary flag indicating whether the fire is actively growing in area at the time of observation.
- **is_closing**: A binary flag indicating whether the fire is currently moving toward the station (positive closing speed).
- **dynamic_x_dist**: An interaction between fire activity and log distance, capturing whether growing fires are also nearby.
- **dist_x_low_res**: An interaction between log distance and the low-resolution flag, flagging distant fires that are also poorly observed.
- **growth_momentum**: The product of initial fire size and radial growth rate, serving as a proxy for the fire's raw energy and potential reach.
- **month_sin**: The sine of the fire's ignition month, encoding seasonal timing in a continuous circular form.
- **month_cos**: The cosine of the fire's ignition month, paired with month_sin to fully represent the time of year.
- **hour_sin**: The sine of the ignition hour, encoding time-of-day patterns in a circular form.
- **hour_cos**: The cosine of the ignition hour, paired with hour_sin to represent the full daily cycle.
- **dow_sin**: The sine of the day of week at ignition, encoding weekly patterns in a circular form.
- **dow_cos**: The cosine of the day of week at ignition, paired with dow_sin for complete weekly cycle representation.
- **align_x_month_sin**: An interaction between fire alignment and seasonal sine, capturing whether directional threat varies by time of year.
- **bearing_x_month_cos**: An interaction between spread bearing cosine and seasonal cosine, encoding how fire direction relates to the season.
- **growth_x_dow_sin**: An interaction between log growth rate and day-of-week sine, capturing whether fire growth patterns differ across the week.

# Modeling (V28) — Seed-Averaged Close-Fire Coxnet

**Architecture:**
- Stage 1: Survival backbone (RSF + GBM + Coxnet) on all 221 examples, 10-fold StratifiedKFold, Nelder-Mead blend (MIN_COXNET_W=0.60)
- Stage 2: Close-fire Coxnet timing model — `CoxnetSurvivalAnalysis(alpha=0.20, l1_ratio=0.5)` on 69 close fires, **5-seed averaged OOF** (seeds=[42,100,200,300,400], 7-fold KFold each)
- Stage 3: Beta blend search [0.00→0.60 step=0.05] + Platt/isotonic calibration

**Key results:** OOF=0.9734  beta=0.55  LB=0.97458  0 violations

**V28 improvement over V21:** Seed averaging (5 seeds) reduces stochastic variance in close-fire Coxnet OOF → more reliable beta calibration (+0.0005 OOF, +0.00020 LB).

**Hard rules:**
- alpha=0.20, l1_ratio=0.5 LOCKED for close-fire Coxnet (V32 lesson: OOF cannot detect regularization overfit at n=69)
- Distance threshold=5000m LOCKED (confirmed optimal)
- MIN_COXNET_W=0.60 LOCKED

In [1]:
import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

In [2]:
SEED           = 42
N_SPLITS       = 10    # backbone cross-validation folds
N_SPLITS_CLOSE = 7     # close-fire Coxnet cross-validation folds
DIST_THRESHOLD = 5_000 # meters — fires within this distance are "close fires"
HORIZONS       = [12, 24, 48, 72]  # prediction time checkpoints in hours
MODEL_NAMES    = ['rsf', 'gbm', 'coxnet']
MIN_COXNET_W   = 0.60  # Coxnet must hold at least 60% of backbone blend weight (LOCKED)
COXNET_ALPHA   = 0.05  # backbone Coxnet regularization strength
CLOSE_ALPHA    = 0.20  # close-fire Coxnet alpha (LOCKED — loosening overfit at n=69, see V32)
CLOSE_L1       = 0.5   # close-fire Coxnet l1_ratio (LOCKED — same reason as above)
CLOSE_SEEDS    = [42, 100, 200, 300, 400]  # 5-seed averaging reduces OOF split variance at n=69
BETA_GRID      = [round(i * 0.05, 2) for i in range(13)]  # blend weight search: 0.00 → 0.60

# Raw columns sourced directly from the dataset
BASE_FEATURES = [
    'num_perimeters_0_5h', 'dt_first_last_0_5h', 'low_temporal_resolution_0_5h',
    'area_first_ha', 'area_growth_rate_ha_per_h', 'log1p_area_first', 'log1p_growth',
    'radial_growth_rate_m_per_h', 'area_growth_rel_0_5h', 'log_area_ratio_0_5h',
    'centroid_speed_m_per_h', 'spread_bearing_sin', 'spread_bearing_cos',
    'centroid_displacement_m', 'closing_speed_m_per_h', 'closing_speed_abs_m_per_h',
    'projected_advance_m', 'dist_fit_r2_0_5h', 'dist_accel_m_per_h2',
    'dist_slope_ci_0_5h', 'alignment_cos', 'alignment_abs',
]

# Features computed from raw columns in the engineer() step below
ENGINEERED = [
    'log_dist_min', 'log_dist_sq', 'dist_close_5k', 'dist_close_30k',
    'tti_h', 'hazard_proxy', 'is_dynamic', 'is_closing', 'dynamic_x_dist',
    'dist_x_low_res', 'growth_momentum',
    'month_sin', 'month_cos', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'align_x_month_sin', 'bearing_x_month_cos', 'growth_x_dow_sin',
]

In [3]:
# Locate SETTINGS.json starting from the current directory and walking up.
# This makes the notebook work regardless of where Jupyter was launched from.
_settings_path = None
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / 'SETTINGS.json').exists():
        _settings_path = _candidate / 'SETTINGS.json'
        break

if _settings_path is None:
    raise FileNotFoundError(
        "SETTINGS.json not found in this directory or any parent. "
        "Place SETTINGS.json in the archive root and update the data paths inside it."
    )

_cfg_dir = _settings_path.parent  # all relative paths in SETTINGS.json resolve from here
with open(_settings_path) as f:
    _cfg = json.load(f)

def _resolve(p):
    """Resolve a path relative to the SETTINGS.json directory, leave absolute paths unchanged."""
    p = Path(p)
    return str((_cfg_dir / p).resolve()) if not p.is_absolute() else str(p)

RAW_DATA_DIR    = _resolve(_cfg['RAW_DATA_DIR'])
TRAIN_PATH      = _resolve(_cfg['TRAIN_DATA_PATH'])
TEST_PATH       = _resolve(_cfg['TEST_DATA_PATH'])
MODEL_PATH      = _resolve(_cfg['MODEL_PATH'])
SUBMISSION_PATH = _resolve(_cfg['SUBMISSION_PATH'])

print(f'SETTINGS loaded from: {_settings_path}')
print(f'TRAIN_PATH:           {TRAIN_PATH}')
print(f'TEST_PATH:            {TEST_PATH}')
print(f'MODEL_PATH:           {MODEL_PATH}')
print(f'SUBMISSION_PATH:      {SUBMISSION_PATH}')

SETTINGS loaded from: /Users/marcelloborromeo/Downloads/WiDS Winner Code/WiDS_2026_bttmapping_Submission_Model/SETTINGS.json
TRAIN_PATH:           /Users/marcelloborromeo/Downloads/WiDS Winner Code/WiDS_2026_bttmapping_Submission_Model/data/train.csv
TEST_PATH:            /Users/marcelloborromeo/Downloads/WiDS Winner Code/WiDS_2026_bttmapping_Submission_Model/data/test.csv
MODEL_PATH:           /Users/marcelloborromeo/Downloads/WiDS Winner Code/WiDS_2026_bttmapping_Submission_Model/model_artifacts (V28).pkl
SUBMISSION_PATH:      /Users/marcelloborromeo/Downloads/WiDS Winner Code/WiDS_2026_bttmapping_Submission_Model/submission (V28_seed_avg).csv


In [4]:
def engineer(df: pd.DataFrame) -> pd.DataFrame:
    """V8-exact 42-feature engineering."""
    df = df.copy()
    eps = 1.0
    raw_dist = df['dist_min_ci_0_5h']

    df['log_dist_min']   = np.log1p(raw_dist)
    df['log_dist_sq']    = df['log_dist_min'] ** 2
    df['dist_close_5k']  = (raw_dist < 5_000).astype(float)
    df['dist_close_30k'] = (raw_dist < 30_000).astype(float)

    closing = df['closing_speed_m_per_h'].clip(lower=0)
    df['tti_h']         = (raw_dist / (closing + eps)).clip(upper=500)
    df['hazard_proxy']  = df['alignment_abs'] * closing / (raw_dist + eps)
    df['is_dynamic']    = (df['area_growth_rate_ha_per_h'] > 0).astype(float)
    df['is_closing']    = (df['closing_speed_m_per_h'] > 0).astype(float)
    df['dynamic_x_dist'] = df['is_dynamic'] * df['log_dist_min']
    df['dist_x_low_res'] = df['log_dist_min'] * df['low_temporal_resolution_0_5h']
    df['growth_momentum'] = df['log1p_area_first'] * df['radial_growth_rate_m_per_h']

    df['month_sin'] = np.sin(2 * np.pi * df['event_start_month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['event_start_month'] / 12)
    df['hour_sin']  = np.sin(2 * np.pi * df['event_start_hour'] / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['event_start_hour'] / 24)
    df['dow_sin']   = np.sin(2 * np.pi * df['event_start_dayofweek'] / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df['event_start_dayofweek'] / 7)

    df['align_x_month_sin']   = df['alignment_abs'] * df['month_sin']
    df['bearing_x_month_cos'] = df['spread_bearing_cos'] * df['month_cos']
    df['growth_x_dow_sin']    = df['log1p_growth'] * df['dow_sin']
    return df

train    = pd.read_csv(TRAIN_PATH)
test     = pd.read_csv(TEST_PATH)
train_fe = engineer(train)
test_fe  = engineer(test)

FEAT_COLS = [f for f in BASE_FEATURES + ENGINEERED if f in train_fe.columns]
print(f'Feature count: {len(FEAT_COLS)}')

y_event = train['event'].values.astype(bool)
y_time  = train['time_to_hit_hours'].values.astype(float)

close_train = train_fe['dist_min_ci_0_5h'].values < DIST_THRESHOLD
close_test  = test_fe['dist_min_ci_0_5h'].values  < DIST_THRESHOLD
print(f'Train close fires: {close_train.sum()}  |  Test close fires: {close_test.sum()}')

pre = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
X_train = pre.fit_transform(train_fe[FEAT_COLS].values)
X_test  = pre.transform(test_fe[FEAT_COLS].values)
y_surv  = Surv.from_arrays(y_event, y_time)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/marcelloborromeo/Downloads/WiDS Winner Code/WiDS_2026_bttmapping_Submission_Model/data/train.csv'

In [ ]:
def surv_fn_to_probs(surv_fns) -> np.ndarray:
    """Convert survival functions to event probabilities at each horizon.
    Returns shape (n_samples, 4) — columns are P(event by 12h/24h/48h/72h)."""
    out = np.zeros((len(surv_fns), len(HORIZONS)))
    for i, sf in enumerate(surv_fns):
        t_min, t_max = sf.domain
        for j, t in enumerate(HORIZONS):
            # P(event by t) = 1 - S(t), clipped to the model's valid time range
            out[i, j] = 1.0 - float(sf(float(np.clip(t, t_min, t_max))))
    return out

def brier_at(p_h, y_evt, y_t, h):
    """Brier score at a single horizon h, with proper censoring exclusion.
    Censored observations (event=0) that were censored before horizon h are
    excluded from scoring since we cannot know their true outcome."""
    hit  = (y_evt == 1) & (y_t <= h)   # fire threatened by horizon h
    excl = (y_evt == 0) & (y_t <  h)   # censored before h — exclude
    keep = ~excl
    if keep.sum() == 0:
        return np.nan
    return float(np.mean((p_h[keep] - hit[keep].astype(float)) ** 2))

def hybrid_score(probs, y_evt, y_t, risk=None):
    """Competition metric: 0.3 * C-index + 0.7 * (1 - WBS).
    WBS = weighted Brier score: 0.3*BS@24h + 0.4*BS@48h + 0.3*BS@72h."""
    wbs  = (0.3 * brier_at(probs[:, 1], y_evt, y_t, 24) +
            0.4 * brier_at(probs[:, 2], y_evt, y_t, 48) +
            0.3 * brier_at(probs[:, 3], y_evt, y_t, 72))
    risk = probs[:, 2] if risk is None else risk  # 48h prob used as ranking score
    try:
        c, *_ = concordance_index_censored(y_evt.astype(bool), y_t, risk)
    except Exception:
        c = 0.5
    return 0.3 * c + 0.7 * (1 - wbs), c, wbs

def platt_fit(raw_h, y_evt, y_t, h):
    """Fit a Platt scaler (logistic regression) to calibrate raw scores at horizon h.
    Same censoring exclusion as brier_at — censored-before-h observations are dropped."""
    hit  = (y_evt == 1) & (y_t <= h)
    excl = (y_evt == 0) & (y_t <  h)
    keep = ~excl
    lr = LogisticRegression(C=1e6, solver='lbfgs', max_iter=2000)
    lr.fit(raw_h[keep].reshape(-1, 1), hit[keep].astype(int))
    return float(lr.coef_[0, 0]), float(lr.intercept_[0])

def iso_fit(raw_h, y_evt, y_t, h):
    """Fit an isotonic regression calibrator at horizon h.
    Used at 48h and 72h where the physical cliff makes a flexible staircase fit appropriate."""
    hit  = (y_evt == 1) & (y_t <= h)
    excl = (y_evt == 0) & (y_t <  h)
    keep = ~excl
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(raw_h[keep], hit[keep].astype(float))
    return iso

def apply_calibration(raw, platts, iso48, iso72):
    """Apply per-horizon calibration and enforce monotonicity.
    Platt (S-curve) at 12h/24h, isotonic (staircase) at 48h/72h.
    cummax enforces P(12h) <= P(24h) <= P(48h) <= P(72h) for every fire."""
    a12, b12 = platts[12]
    a24, b24 = platts[24]
    out = np.column_stack([
        expit(a12 * raw[:, 0] + b12),
        expit(a24 * raw[:, 1] + b24),
        iso48.predict(raw[:, 2]),
        iso72.predict(raw[:, 3]),
    ])
    # Enforce monotonicity: cumulative max ensures probabilities never decrease over time
    return np.clip(np.maximum.accumulate(out, axis=1), 0.0, 1.0)

In [ ]:
def build_rsf(n_estimators=200):
    return RandomSurvivalForest(
        n_estimators=n_estimators, max_depth=4, min_samples_leaf=10,
        max_features='sqrt', n_jobs=-1, random_state=SEED,
    )

def build_gbm():
    return GradientBoostingSurvivalAnalysis(
        n_estimators=150, learning_rate=0.04, max_depth=2,
        subsample=0.8, min_samples_leaf=15, random_state=SEED,
    )

def build_coxnet_backbone():
    return CoxnetSurvivalAnalysis(
        alphas=[COXNET_ALPHA], l1_ratio=0.5,
        fit_baseline_model=True, max_iter=10_000,
    )

def build_coxnet_close():
    """Close-fire timing Coxnet — alpha=0.20, l1=0.5 LOCKED."""
    return CoxnetSurvivalAnalysis(
        alphas=[CLOSE_ALPHA], l1_ratio=CLOSE_L1,
        fit_baseline_model=True, max_iter=10_000,
    )

In [ ]:
print('Stage 1: Backbone survival OOF (10-fold StratifiedKFold)...')
np.random.seed(SEED)

n   = len(X_train)
oof = {name: np.zeros((n, len(HORIZONS))) for name in MODEL_NAMES}

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fold, (tr_i, va_i) in enumerate(skf.split(X_train, y_event.astype(int))):
    X_tr, X_va = X_train[tr_i], X_train[va_i]
    y_tr = y_surv[tr_i]

    m = build_rsf()
    m.fit(X_tr, y_tr)
    oof['rsf'][va_i] = surv_fn_to_probs(m.predict_survival_function(X_va))

    m = build_gbm()
    m.fit(X_tr, y_tr)
    oof['gbm'][va_i] = surv_fn_to_probs(m.predict_survival_function(X_va))

    try:
        m = build_coxnet_backbone()
        m.fit(X_tr, y_tr)
        oof['coxnet'][va_i] = surv_fn_to_probs(m.predict_survival_function(X_va))
    except Exception:
        # Coxnet can fail to converge on small folds; fall back to RSF predictions
        oof['coxnet'][va_i] = oof['rsf'][va_i].copy()

print('  Backbone OOF complete.')

In [ ]:
print('Nelder-Mead 3-way blend (MIN_COXNET_W=0.60)...')

coxnet_i = MODEL_NAMES.index('coxnet')
rng      = np.random.RandomState(SEED + 1)

# Use 11 starting points to avoid local optima: equal weights, 9 random Dirichlet draws,
# and one Coxnet-heavy initialization based on V8's known good weights
starts   = [np.ones(3) / 3] + [rng.dirichlet(np.ones(3)) for _ in range(9)]
v8like   = np.zeros(3); v8like[coxnet_i] = 0.79; v8like[1] = 0.21  # V8 prior: Coxnet-dominant
starts.append(v8like)

def blend_obj(w_raw):
    # Normalize raw weights to sum to 1, then compute hybrid score (negated for minimization)
    w = np.abs(w_raw); s = w.sum()
    if s < 1e-9: return 1.0
    w /= s
    b    = sum(w[i] * oof[MODEL_NAMES[i]] for i in range(3))
    wbs  = (0.3 * brier_at(b[:, 1], y_event, y_time, 24) +
            0.4 * brier_at(b[:, 2], y_event, y_time, 48) +
            0.3 * brier_at(b[:, 3], y_event, y_time, 72))
    try:
        c, *_ = concordance_index_censored(y_event.astype(bool), y_time, b[:, 2])
    except Exception:
        c = 0.5
    return -(0.3 * c + 0.7 * (1 - wbs))

# Keep only solutions where Coxnet weight >= MIN_COXNET_W (LOCKED constraint)
best_obj = np.inf; best_w = None
for w0 in starts:
    res = minimize(blend_obj, w0, method='Nelder-Mead',
                   options={'maxiter': 5000, 'xatol': 1e-7, 'fatol': 1e-8})
    w = np.abs(res.x); w /= w.sum()
    if res.fun < best_obj and w[coxnet_i] >= MIN_COXNET_W:
        best_obj = res.fun; best_w = w.copy()

if best_w is None:
    # Fallback: no run satisfied the Coxnet constraint — use V8's validated weights
    best_w = np.array([0.044, 0.178, 0.778])
    print('  WARNING: constraint not met, using V8 fallback weights')

weights       = dict(zip(MODEL_NAMES, best_w))
raw_blend_oof = sum(weights[nm] * oof[nm] for nm in MODEL_NAMES)

print(f'  Weights: coxnet={weights["coxnet"]:.3f}  gbm={weights["gbm"]:.3f}  rsf={weights["rsf"]:.3f}')
h_back, c_back, wbs_back = hybrid_score(raw_blend_oof, y_event, y_time)
print(f'  Backbone OOF: hybrid={h_back:.4f}  C={c_back:.4f}  WBS={wbs_back:.4f}')

In [ ]:
print('Stage 2: Close-fire Coxnet OOF (5-seed average, 7-fold KFold)...')

X_close      = X_train[close_train]
y_surv_close = y_surv[close_train]
n_close      = close_train.sum()

# Run 7-fold CV 5 times with different seeds and average the OOF predictions.
# This reduces variance from random split assignment at n=69 (small sample).
seed_oofs = []
for s in CLOSE_SEEDS:
    kf       = KFold(n_splits=N_SPLITS_CLOSE, shuffle=True, random_state=s)
    seed_oof = np.zeros((n_close, len(HORIZONS)))
    for tr, va in kf.split(X_close):
        try:
            m = build_coxnet_close()
            m.fit(X_close[tr], y_surv_close[tr])
            seed_oof[va] = surv_fn_to_probs(m.predict_survival_function(X_close[va]))
        except Exception:
            # Coxnet can fail on very small folds; fall back to backbone predictions
            seed_oof[va] = raw_blend_oof[close_train][va]
    seed_oofs.append(seed_oof)
    print(f'  Seed {s} done.')

close_cox_oof = np.mean(seed_oofs, axis=0)
print(f'  Close-fire Coxnet OOF complete ({n_close} fires, {len(CLOSE_SEEDS)} seeds).')

In [ ]:
print('Stage 3: Beta blend search + Platt/isotonic calibration...')

best_h    = -np.inf
best_beta = 0
best_cals = None
best_c    = 0
best_wbs  = 0

for beta in BETA_GRID:
    mixed = raw_blend_oof.copy()
    mixed[close_train] = (1 - beta) * raw_blend_oof[close_train] + beta * close_cox_oof

    platts = {
        12: platt_fit(mixed[:, 0], y_event, y_time, 12),
        24: platt_fit(mixed[:, 1], y_event, y_time, 24),
    }
    iso48 = iso_fit(mixed[:, 2], y_event, y_time, 48)
    iso72 = iso_fit(mixed[:, 3], y_event, y_time, 72)

    cal = apply_calibration(mixed, platts, iso48, iso72)
    h, c, wbs = hybrid_score(cal, y_event, y_time, risk=mixed[:, 2])

    if h > best_h:
        best_h    = h
        best_beta = beta
        best_c    = c
        best_wbs  = wbs
        best_cals = {'platts': platts, 'iso48': iso48, 'iso72': iso72}

print(f'  Best beta: {best_beta}')
print(f'  OOF hybrid: {best_h:.4f}  C: {best_c:.4f}  WBS: {best_wbs:.4f}')
print(f'  V28 reference: OOF=0.9734  LB=0.97458')

Stage 3: Beta blend search + Platt/isotonic calibration...
  Best beta: 0.55
  OOF hybrid: 0.9734  C: 0.9448  WBS: 0.0143
  V28 reference: OOF=0.9734  LB=0.97458


In [ ]:
versions = {
    'V8  (LB=0.97090)': {'c': 0.9403, 'wbs': 0.0148, 'h': 0.9717},
    'V21 (LB=0.97438)': {'c': 0.9448, 'wbs': 0.0143, 'h': 0.9729},
    'V28 (LB=0.97458)': {'c': 0.9448, 'wbs': 0.0143, 'h': 0.9734},
    'This run':          {'c': best_c,  'wbs': best_wbs, 'h': best_h},
}
print(f'{"Version":<22} {"Hybrid":>8} {"C-index":>9} {"WBS":>8}')
print('-' * 52)
for vname, m in versions.items():
    print(f'{vname:<22} {m["h"]:>8.4f} {m["c"]:>9.4f} {m["wbs"]:>8.4f}')

Version                  Hybrid   C-index      WBS
----------------------------------------------------
V8  (LB=0.97090)         0.9717    0.9403   0.0148
V21 (LB=0.97438)         0.9729    0.9448   0.0143
V28 (LB=0.97458)         0.9734    0.9448   0.0143
This run                 0.9734    0.9448   0.0143


In [ ]:
print('Training final models on all 221 examples...')

m_rsf = build_rsf(n_estimators=400)
m_rsf.fit(X_train, y_surv)

m_gbm = build_gbm()
m_gbm.fit(X_train, y_surv)

m_cox_backbone = build_coxnet_backbone()
m_cox_backbone.fit(X_train, y_surv)

final_models = {'rsf': m_rsf, 'gbm': m_gbm, 'coxnet': m_cox_backbone}
print('  Backbone final models trained.')

m_cox_close = build_coxnet_close()
m_cox_close.fit(X_close, y_surv_close)
print('  Close-fire Coxnet final model trained.')

Training final models on all 221 examples...


  Backbone final models trained.
  Close-fire Coxnet final model trained.


In [ ]:
print('Generating test predictions...')

test_raw = sum(
    weights[nm] * surv_fn_to_probs(final_models[nm].predict_survival_function(X_test))
    for nm in MODEL_NAMES
)

X_test_close      = X_test[close_test]
close_test_probs  = surv_fn_to_probs(m_cox_close.predict_survival_function(X_test_close))

test_mixed = test_raw.copy()
test_mixed[close_test] = (
    (1 - best_beta) * test_raw[close_test] + best_beta * close_test_probs
)

test_cal = apply_calibration(test_mixed, best_cals['platts'], best_cals['iso48'], best_cals['iso72'])
print('  Test predictions generated.')

Generating test predictions...
  Test predictions generated.


In [ ]:
print('=== Submission Monotonicity Check ===')
v_12_24 = int((test_cal[:, 0] > test_cal[:, 1]).sum())
v_24_48 = int((test_cal[:, 1] > test_cal[:, 2]).sum())
v_48_72 = int((test_cal[:, 2] > test_cal[:, 3]).sum())
total_v = v_12_24 + v_24_48 + v_48_72
print(f'  12→24h violations: {v_12_24}')
print(f'  24→48h violations: {v_24_48}')
print(f'  48→72h violations: {v_48_72}')
print(f'  Total: {total_v}')
assert total_v == 0, f'Monotonicity violated! {total_v} violations.'

sub = pd.DataFrame({
    'event_id': test['event_id'],
    'prob_12h':  test_cal[:, 0],
    'prob_24h':  test_cal[:, 1],
    'prob_48h':  test_cal[:, 2],
    'prob_72h':  test_cal[:, 3],
})
sub.to_csv(SUBMISSION_PATH, index=False)
print(f'  Saved: {SUBMISSION_PATH}')
print(f'  Shape: {sub.shape}')
print(sub.head())

In [ ]:
artifacts = {
    'version':        'V28',
    'feat_cols':      FEAT_COLS,
    'preprocessor':   pre,
    'weights':        weights,
    'best_beta':      best_beta,
    'best_cals':      best_cals,
    'final_models':   final_models,
    'close_model':    m_cox_close,
    'close_train':    close_train,
    'close_test':     close_test,
    'close_seeds':    CLOSE_SEEDS,
    'oof_hybrid':     best_h,
    'oof_c':          best_c,
    'oof_wbs':        best_wbs,
    'lb_score':       0.97458,
}
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(artifacts, f)
print(f'Artifacts saved: {MODEL_PATH}')

In [ ]:
print('=' * 60)
print('V28 FINAL SUMMARY')
print('=' * 60)
print(f'  Feature count:     {len(FEAT_COLS)}')
print(f'  Close-fire seeds:  {CLOSE_SEEDS}')
print(f'  Close alpha/l1:    {CLOSE_ALPHA} / {CLOSE_L1}  (LOCKED)')
print(f'  Best beta:         {best_beta}')
print(f'  OOF hybrid:        {best_h:.4f}')
print(f'  OOF C-index:       {best_c:.4f}')
print(f'  OOF WBS:           {best_wbs:.4f}')
print(f'  LB score:          0.97458')
print(f'  Violations:        0')
print('=' * 60)

V28 FINAL SUMMARY
  Feature count:     42
  Close-fire seeds:  [42, 100, 200, 300, 400]
  Close alpha/l1:    0.2 / 0.5  (LOCKED)
  Best beta:         0.55
  OOF hybrid:        0.9734
  OOF C-index:       0.9448
  OOF WBS:           0.0143
  LB score:          0.97458
  Violations:        0
